# KrishiDisha assistant - frozen-set evaluation on Kaggle (T4)

Scores the rules bot and the fine-tunes (v0, v1 from the private GGUF dataset; v2 from the training kernel's output) on the frozen 606-example eval set through the real `AgriAssistant`, one system at a time, with the harness's per-example resume. Ollama runs inside the kernel on the T4. Private; weights never leave Kaggle.


In [ ]:
# Source comes from the private Kaggle dataset (git archive of the lab repo), not from GitHub.
# Kaggle extracts uploaded zips, so the tree is at .../krishidisha-lab-src/src/krishidisha; fall back to src.zip.
import glob, os, shutil, zipfile
shutil.rmtree("/tmp/krishidisha", ignore_errors=True)
_dirs = glob.glob("/kaggle/input/**/krishidisha-lab-src/src/krishidisha", recursive=True)
if _dirs:
    shutil.copytree(_dirs[0], "/tmp/krishidisha")
    _ver = os.path.join(os.path.dirname(os.path.dirname(_dirs[0])), "VERSION.txt")
else:
    _zip = glob.glob("/kaggle/input/**/krishidisha-lab-src/**/src.zip", recursive=True)[0]
    zipfile.ZipFile(_zip).extractall("/tmp")
    _ver = os.path.join(os.path.dirname(_zip), "VERSION.txt")
print("source version:", open(_ver).read().strip() if os.path.exists(_ver) else "unknown", "->", os.listdir("/tmp/krishidisha")[:8])

import os, glob, subprocess, time, shutil
W = "/kaggle/working"
os.chdir("/tmp/krishidisha")
os.environ["TRANSLATION_BACKEND"] = "none"
os.environ["KB_EMBEDDING_MODEL"] = ""
os.makedirs(f"{W}/eval", exist_ok=True)
ggufs = {
    "krishidisha-v0": glob.glob("/kaggle/input/**/krishidisha-llm-gguf/**/krishidisha-v0-q4_k_m.gguf", recursive=True),
    "krishidisha-v1": glob.glob("/kaggle/input/**/krishidisha-llm-gguf/**/krishidisha-v1-q4_k_m.gguf", recursive=True),
    "krishidisha-v2": glob.glob("/kaggle/input/**/krishidisha-assistant-qlora-training/**/gguf/krishidisha-q4_k_m.gguf", recursive=True),
}
ggufs = {k: v[0] for k, v in ggufs.items() if v}
print("ggufs:", ggufs)
assert ggufs, "no GGUF found: attach krishidisha-llm-gguf and the training kernel output"


In [ ]:
CMD = "pip install -q flask flask-sqlalchemy python-dotenv openai anthropic reportlab jsonschema 2>&1 | tail -1"
!{CMD}
CMD = "apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1); which zstd"
!{CMD}
CMD = "curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3; which ollama && ollama --version"
!{CMD}


In [ ]:
srv = subprocess.Popen(["ollama", "serve"], stdout=open(f"{W}/ollama.log", "w"), stderr=subprocess.STDOUT,
                       env=dict(os.environ, OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="30m"))
import urllib.request
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
tpl = open("models/llm/Modelfile.qwen", encoding="utf-8").read()
for tag, path in ggufs.items():
    mf = f"/tmp/Modelfile.{tag}"
    open(mf, "w", encoding="utf-8").write(tpl.replace("FROM ./MODEL.gguf", f"FROM {path}"))
    r = subprocess.run(["ollama", "create", tag, "-f", mf], capture_output=True, text=True)
    print(tag, "->", (r.stdout + r.stderr).strip().splitlines()[-1][:80])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)


In [ ]:
SYSTEMS = ["rules"] + [f"ollama:{t}" for t in ggufs]
for spec in SYSTEMS:
    safe = spec.replace(":", "_")
    CMD = f"python -u -m ml.llm.eval_llm --eval data/llm/eval.jsonl --systems {spec} --out {W}/eval/{safe}.md 2>&1 | grep -vE 'INFO|Warning|Mandi' | tee {W}/eval/{safe}.log | tail -5"
    print("====", spec, flush=True)
    !{CMD}
CMD = f"python scripts/merge_llm_eval.py {W}/eval/*.json --out {W}/eval/llm_eval_merged.md"
!{CMD}


In [ ]:
for t in ggufs:
    subprocess.run(["ollama", "rm", t], capture_output=True)
srv.terminate()
print(open(f"{W}/eval/llm_eval_merged.md", encoding="utf-8").read())
